# L3: Voice for your Agent

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em> and select <em>"Notebook (.ipynb)"</em>.</p>
</div>

<br>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different run results:</b> The output generated by AI models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

**Two Lines of Code: Give Your LLM a Voice**

Most teams already have an LLM agent in production. This lesson shows you the lowest-effort path to giving it a voice — without rewriting any of it.

The pattern is **AI Agent Integration mode**. Vocal Bridge sits in front as a thin voice layer; every spoken question is forwarded to *your* endpoint as a `query_agent` action; you respond with a string; VB speaks it.

Under the hood:

1. User speaks → VB transcribes
2. VB fires `query_agent` over the data channel
3. Your endpoint runs Claude (or GPT, or Gemini, or your in-house model)
4. You return a string → VB speaks it


## Setup

You'll spin up a tiny FastAPI server in a background cell so the in-notebook widget has somewhere to forward queries. In production this is just another route in your backend.


In [ ]:
import json
import os
from pathlib import Path

from helpers import (
    load_env, mint_token, vb, append_to_env, voice_widget,
    start_query_server, stop_query_server,
)

load_env()
assert os.getenv("VOCAL_BRIDGE_API_KEY"), "Set VOCAL_BRIDGE_API_KEY in .env"
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env"
print("Both API keys loaded ✓")


### What's in `helpers.py`

| Helper | What it does | What it wraps |
| --- | --- | --- |
| `mint_token(agent_id)` | Returns a voice session token | one `requests.post()` to `/api/v1/token` |
| `vb(*args)` | Runs a `vb` CLI command | `subprocess.run(["vb", …])` + JSON parsing |
| `start_query_server(handler)` | Spins up a tiny HTTP server in a background thread | a 5-line FastAPI route the widget can POST to |
| `voice_widget(token, query_url)` | Renders an in-notebook React widget | a small component using `useAIAgent({ onQuery })` |


## Step 1 — Build the agent (text-only first)

This is the function VB will call for every spoken turn. You use Claude with web search enabled — the same call you'd already have in your backend.

A typical Claude Q&A takes 1–6 seconds (longer if web search runs). VB fills the silence with natural noises while it waits — but your prompt also instructs the agent to say a short *bridge line* ("hang on — checking…") the moment a query is delegated. That's the difference between feeling fast and feeling broken.


In [ ]:
import os
import anthropic

claude = anthropic.Anthropic()

def answer(query: str) -> str:
    msg = claude.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=400,
        tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 2}],
        messages=[{"role": "user", "content": query}],
    )
    parts = [b.text for b in msg.content if getattr(b, "type", None) == "text"]
    return " ".join(parts).strip() or "I don't have an answer for that."

# Quick smoke test
print(answer("Say hi in five words."))

## Step 2 — Expose the agent over HTTP

The widget needs an HTTP endpoint to POST queries to. You start a tiny FastAPI server in the background — it exposes `POST /query` which calls `answer()` above.

`start_query_server()` returns the URL to point the widget at. Under the hood it's just FastAPI — the entire route is five lines:

```python
from fastapi import FastAPI

app = FastAPI()

@app.post("/query")
def query(req: dict):
    return {"response": answer(req["query"])}
```

In production this is just another route in your existing backend. The notebook spins it up locally so the in-page widget has somewhere to call.


In [ ]:
server_url = start_query_server(answer, port=8765)
print(f"Query endpoint up: {server_url}")

## Step 3 — Tell VB about your agent

`ai-agent.json` tells the VB agent two things:
- **enabled** — turn AI Agent mode on
- **description** — what your agent is good at (guides VB on when to delegate vs. answer itself)
- **verbatim** — `false` lets VB lightly polish phrasing for natural voice delivery; `true` reads your response word-for-word


In [ ]:
print(Path("agents/voice-agent/prompt.md").read_text())

In [ ]:
print(Path("agents/voice-agent/ai-agent.json").read_text())

## Step 4 — Create the VB agent

AI Agent mode and Background System are mutually exclusive in VB. You pass `--background-enabled false` at create time so the AI Agent config sticks.

You also leave the greeting empty — Claude will own the opening line.


In [ ]:
import os

agent_id = os.environ.get("VOCAL_BRIDGE_AGENT_ID_L3", "").strip()
if agent_id:
    print(f"Using pre-provisioned agent: {agent_id}")
else:
    result = vb(
        "agent", "create",
        "--name", "lesson-3-voice-agent",
        "--style", "Chatty",
        "--prompt-file", "agents/voice-agent/prompt.md",
        "--ai-agent-file", "agents/voice-agent/ai-agent.json",
        "--background-enabled", "false",
        "--greeting", "",
        "--deploy-targets", "web",
        json_output=True,
    )
    agent = result["agent"]
    agent_id = agent["id"]
    append_to_env("VOCAL_BRIDGE_AGENT_ID_L3", agent_id)
    print(f"Created '{agent['name']}' (AI Agent mode)")
    print(f"  id: {agent_id}")

## Step 5 — Mint a token + render the widget

The widget connects to VB, listens for `query_agent`, POSTs to your local endpoint, and replies with `agent_response`. Everything else (transcript, audio) is handled by VB.

Click Connect, then try asking something Claude needs the web for:
- *"What was NVIDIA's closing price yesterday?"*
- *"Who won the F1 race last weekend?"*

### The pattern that powers all of this

The whole "give my LLM a voice" wiring is **one React hook** — `useAIAgent({ onQuery })`. Every spoken question arrives as `query`; whatever string you return, VB speaks:

```jsx
import { useAIAgent } from '@vocalbridgeai/react';

function VoiceLayer() {
  useAIAgent({
    onQuery: async (query) => {
      // VB sends every spoken question here.
      const res = await fetch('/api/query', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ query }),
      });
      const { response } = await res.json();
      return response;  // VB will speak this back to the user.
    },
  });
}
```

That's the entire client-side delta from text-chat to voice-chat. Works with Claude, GPT, Gemini, your fine-tuned model, your RAG stack — anything that returns text.


In [ ]:
token = mint_token(agent_id, participant_name="Notebook Learner")
from IPython.display import HTML
HTML(voice_widget(token, query_url=server_url))


## Step 6 — Read the call log

When you end the session, VB stores a structured log of every turn and every action. You can pull the most recent one from the CLI:

In [ ]:
vb("agent", "use", agent_id)
logs = vb("logs", "--limit", "1", json_output=True)
if not logs:
    print("No calls yet — try the widget above first.")
else:
    print(vb("logs", "show", logs['sessions'][0]['id']))

## Step 7 — Stop the server

When you're done, shut the FastAPI server down so it doesn't keep the port held:

In [ ]:
stop_query_server()
print("Query server stopped.")


## What just happened

You wired a Claude-powered Q&A function into a voice agent in three pieces:

1. **VB AI Agent mode** — turns on with one CLI flag (`--ai-agent-enabled true` in production)
2. **Your endpoint** — any HTTP route that takes a query and returns text
3. **Two lines on the client** — `useAIAgent({ onQuery })` in React, or the data-channel handler we use here

Same pattern works with GPT, Gemini, your fine-tuned model, your RAG stack — anything that takes text in, returns text out.

## What's next

**Lesson 4** flips the relationship. Instead of voice forwarding to your LLM, your LLM gets *voice as a tool* — `make_phone_call` and `start_voice_conversation` in its tool palette.
